In [ ]:
import cv2
import numpy as np

# --- 1. Load two consecutive images (must be in grayscale) ---
img1 = cv2.imread('/Users/edwardamoah/Downloads/image3.png')
img2 = cv2.imread('/Users/edwardamoah/Downloads/image4.png') 

if img1 is None or img2 is None:
    print("Error: Could not load one or both images. Ensure image2.png exists.")
else:
    # Convert to grayscale
    prev_gray = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    frame_gray = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    # =============================================================
    # FIX 1: Explicitly resize images to ensure matching dimensions for flow calculation
    # We match everything to the size of the first grayscale image
    h, w = prev_gray.shape
    frame_gray = cv2.resize(frame_gray, (w, h), interpolation=cv2.INTER_AREA)
    img2 = cv2.resize(img2, (w, h), interpolation=cv2.INTER_AREA)
    img1 = cv2.resize(img1, (w, h), interpolation=cv2.INTER_AREA) # Also resize img1 for consistency
    # =============================================================

    # --- 2. Define parameters for feature detection ---
    feature_params = dict(maxCorners=100,
                          qualityLevel=0.3,
                          minDistance=7,
                          blockSize=7)
    
    # Detect initial points (corners) in the first image
    p0 = cv2.goodFeaturesToTrack(prev_gray, mask=None, **feature_params)

    # --- 3. Define parameters for the Lucas-Kanade optical flow algorithm ---
    lk_params = dict(winSize=(15, 15),
                     maxLevel=2,
                     criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

    # =============================================================
    # FIX 2: Create the mask *after* resizing img1/img2 to ensure size matches for cv2.add()
    # Mask uses the final dimensions/channels of the color images (img1 or img2)
    mask = np.zeros_like(img1) 
    # =============================================================
    
    # --- 4. Calculate Optical Flow ---
    p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, frame_gray, p0, None, **lk_params)

    # --- 5. Select and draw good points/tracks ---
    if p1 is not None:
        good_new = p1[st == 1]
        good_old = p0[st == 1]

        for i, (new, old) in enumerate(zip(good_new, good_old)):
            a, b = new.ravel()
            c, d = old.ravel()
            # Draw a line connecting the old point to the new point on the mask
            mask = cv2.line(mask, (int(c), int(d)), (int(a), int(b)), (0, 255, 0), 2)
            # Draw a circle at the new position on the second image
            img2 = cv2.circle(img2, (int(a), int(b)), 5, (0, 255, 0), -1)

        # Combine the original image with the flow lines (now both are same size/channels)
        img_flow = cv2.add(img2, mask)

        # Display the result
        cv2.imshow('Optical Flow Tracks', img_flow)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    else:
        print("No features were successfully tracked.")
